# Agente de Gestión de Inventario Aeronáutico con MCP Remoto (HTTP)\n\n## Descripción\n\nEn este ejercicio, construirás un **sistema de agentes con un servidor MCP remoto auto-hospedado**, que expondrá herramientas a través de HTTP. Un agente cliente se conectará a este servidor para responder preguntas sobre inventario y consumo de componentes aeronáuticos.\n\n### Arquitectura MCP Remoto\n```\nPregunta sobre inventario\n    ↓\n[InventoryAgent]\n    ↓\n[MCPStreamableHTTPTool] → http://<codespaces-url>:8000\n    ↓\nServidor Python (remote_mcp_server.py)\n    ↓\n[Herramientas locales]\n    ├→ get_inventory_levels()\n    └→ get_weekly_sales()\n    ↓\nAnálisis y respuesta\n```\n\n### Ventajas de MCP Remoto\n- **Centralización**: Las herramientas se pueden alojar en un único servidor y ser consumidas por múltiples agentes.\n- **Escalabilidad**: El servidor puede escalar de forma independiente a los agentes.\n- **Accesibilidad**: Cualquier agente con acceso a la red puede utilizar las herramientas.

### 1. Crear el servidor MCP remoto (HTTP)\n\nPrimero, creamos un archivo `remote_mcp_server.py`. Este script utiliza `FastMCP` para definir las herramientas y `uvicorn` para servirlas a través de HTTP. Este archivo se debe ejecutar como un proceso separado en una terminal.

In [ ]:
server_code = """\nfrom mcp.server.fastmcp import FastMCP\nimport uvicorn\n\nmcp = FastMCP(\"AeroInventory\")\n\n@mcp.tool()\ndef get_inventory_levels() -> dict:\n    \"\"\"Devuelve el inventario actual de componentes aero clave en las plantas.\"\"\"\n    return {\n        \"Álabe de turbina (AP)\": 6,\n        \"Carcasa del compresor\": 8,\n        \"Conjunto de sello\": 28,\n        \"Álabe guía de tobera\": 5,\n        \"Carcasa del rodamiento\": 12,\n        \"Eje\": 9,\n        \"Difusor\": 30,\n        \"Revestimiento del combustor\": 3,\n        \"Bastidor frontal\": 17,\n        \"Carcasa del ventilador\": 45,\n    }\n\n@mcp.tool()\ndef get_weekly_sales() -> dict:\n    \"\"\"Devuelve el consumo semanal por órdenes de trabajo de la última semana.\"\"\"\n    return {\n        \"Álabe de turbina (AP)\": 22,\n        \"Carcasa del compresor\": 18,\n        \"Conjunto de sello\": 3,\n        \"Álabe guía de tobera\": 2,\n        \"Carcasa del rodamiento\": 14,\n        \"Eje\": 19,\n        \"Difusor\": 4,\n        \"Revestimiento del combustor\": 1,\n        \"Bastidor frontal\": 13,\n        \"Carcasa del ventilador\": 17,\n    }\n\nif __name__ == \"__main__\":\n    uvicorn.run(mcp, host=\"0.0.0.0\", port=8000)\n\n\"\"\"\nConexión del agente\n\nEl `ChatAgent` se conectará al servidor remoto utilizando `MCPStreamableHTTPTool`, \nque apunta a la URL donde el servidor MCP está expuesto.\n\n"\"\"\n"""\n\nprint(\"Creando el archivo del servidor MCP remoto...\")\nwith open(\"remote_mcp_server.py\", \"w\") as f:\n    f.write(server_code)\nprint(\"Archivo 'remote_mcp_server.py' creado con éxito.\")

### 2. Ejecutar el Servidor en GitHub Codespaces\n\n**Pasos:**\n1. Abre una nueva terminal en GitHub Codespaces (`Terminal > New Terminal`).\n2. Ejecuta el servidor con el siguiente comando: `python remote_mcp_server.py`\n3. Codespaces detectará automáticamente que el puerto `8000` está siendo utilizado y te mostrará una notificación para hacer el reenvío de puertos (port forwarding).\n4. Haz clic en **Open in Browser** o copia la URL proporcionada. Esta será la dirección de tu servidor MCP.\n5. Pega esa URL en la variable `CODESPACES_FORWARDED_URL` de la celda de código más abajo.

### 3. Cargar librerías y variables de entorno para el cliente

In [ ]:
import asyncio\nimport os\nfrom dotenv import load_dotenv, find_dotenv\nfrom agent_framework import ChatAgent, MCPStreamableHTTPTool\nfrom agent_framework.openai import OpenAIChatClient\n\nload_dotenv(find_dotenv(usecwd=True))\n\n# Configuración del cliente de GitHub Models\nMODEL_NAME = os.getenv(\"GITHUB_MODEL\", \"openai/gpt-4o\")

### 4. Crear y ejecutar el agente cliente

In [ ]:
async def query_inventory(prompt, server_url):\n    if not server_url or 'YOUR_URL_HERE' in server_url:\n        print(\"Error: Por favor, establece la variable CODESPACES_FORWARDED_URL con la URL de tu servidor.\")\n        return\n\n    print(f\"Conectando al servidor MCP remoto en {server_url}...\")\n    \n    client = OpenAIChatClient(\n        model_id=MODEL_NAME,\n        api_key=os.environ[\"GITHUB_TOKEN\"],\n        base_url=\"https://models.github.ai/inference\"\n    )\n    \n    async with (\n        MCPStreamableHTTPTool(\n            name=\"aeroinventory\",\n            url=server_url,\n        ) as mcp_server,\n        ChatAgent(\n            chat_client=client,\n            name=\"InventoryAgent\",\n            instructions=(\n                \"Eres un asistente de gestión de inventario aeronáutico que puede consultar niveles de stock y ventas semanales a través de un servidor remoto.\"\n            ),\n        ) as agent,\n    ):\n        print(\"Conectado al servidor MCP remoto.\")\n        print(f\"Ejecutando consulta de inventario: {prompt}\")\n        result = await agent.run(prompt, tools=mcp_server)\n        print(\"Resultado de la consulta:\")\n        print(result)

In [ ]:
# Pega aquí la URL que obtuviste de GitHub Codespaces\nCODESPACES_FORWARDED_URL = \"YOUR_URL_HERE/api/mcp\"  # Ejemplo: https://tu-codespace-url-8000.preview.app.github.dev/api/mcp\n\nprint(\"Iniciando análisis de inventario...\")\nuser_prompt = \"Lista los componentes con inventario < 15 y su consumo semanal.\"\nawait query_inventory(user_prompt, CODESPACES_FORWARDED_URL)\nprint(\"Análisis de inventario completado.\")

In [ ]:
# Pega aquí la URL que obtuviste de GitHub Codespaces\nCODESPACES_FORWARDED_URL = \"YOUR_URL_HERE/api/mcp\"  # Asegúrate de que esta URL sea la correcta\n\nprint(\"Iniciando análisis de inventario...\")\nuser_prompt = \"¿Qué piezas podrían agotarse esta semana considerando el consumo semanal y el inventario actual?\"\nawait query_inventory(user_prompt, CODESPACES_FORWARDED_URL)\nprint(\"Análisis de inventario completado.\")